# Starfelt — ResNet-18 on CIFAR-10
Real model. Real data. Drive-backed checkpoints.

> Runtime → Change runtime type → T4 GPU before running.

In [ ]:
# Cell 1 — Install
!pip install -q git+https://github.com/victorachede/starfelt.git
!pip install -q torchvision

In [ ]:
# Cell 2 — Imports and data
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from starfelt import Trainer
from starfelt.notebook import StarfeltDisplay

train_tf = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])
val_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

train_set = torchvision.datasets.CIFAR10(root='./data', train=True,  download=True, transform=train_tf)
val_set   = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=val_tf)

train_loader = DataLoader(train_set, batch_size=128, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_set,   batch_size=256, shuffle=False, num_workers=2, pin_memory=True)
print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')

In [ ]:
# Cell 3 — Model and optimizer
model = torchvision.models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, 10)

optimizer = torch.optim.SGD(
    model.parameters(), lr=0.1,
    momentum=0.9, weight_decay=5e-4
)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# Cell 4 — Train with Starfelt
display = StarfeltDisplay()

trainer = Trainer(
    model=model,
    optimizer=optimizer,
    train_loader=train_loader,
    loss_fn=nn.CrossEntropyLoss(),
    val_loader=val_loader,
    epochs=10,
    amp=True,
    early_stop_patience=4,
    checkpoint_every_epochs=1,
    auto_mount_drive=True,
    notebook_display=display,
)

result = trainer.fit()

In [ ]:
# Cell 5 — Results
print(f'Run ID:         {result.run_id}')
print(f'Epochs:         {result.epochs_completed}')
print(f'Final loss:     {result.final_loss:.4f}')
print(f'Final val loss: {result.final_val_loss}')
print(f'Cost tracked:   ${result.cost_usd:.4f}')
print(f'Checkpoint:     {result.checkpoint_path}')
print(f'Stopped early:  {result.stopped_early}')

run_id = result.run_id
print(f'Save this for resume: {run_id}')

In [ ]:
# Cell 6 — Resume after disconnect
import os
# run_id = 'paste-your-run-id-here'  # uncomment if starting fresh session

ckpt = f'/content/drive/MyDrive/.starfelt/checkpoints/{run_id}/latest.pt'
os.environ['STARFELT_RESUME_FROM'] = ckpt
os.environ['STARFELT_RUN_ID'] = run_id
print(f'Resuming from: {ckpt}')

resume_display = StarfeltDisplay()
resume_trainer = Trainer(
    model=model,
    optimizer=optimizer,
    train_loader=train_loader,
    loss_fn=nn.CrossEntropyLoss(),
    val_loader=val_loader,
    epochs=10,
    amp=True,
    auto_mount_drive=True,
    notebook_display=resume_display,
)
resumed = resume_trainer.fit()
print(f'Resumed — completed {resumed.epochs_completed} epochs')